### Data Class

In [ ]:
import random

class Sentiment:
    NEGATIVE = "NEGATIVE"
    NEUTRAL = "NEUTRAL"
    POSITIVE = "POSITIVE"

class Review:
    def __init__(self, text, score):
        self.text = text
        self.score = score
        self.sentiment = self.get_sentiment()
        
    def get_sentiment(self):
        if self.score <= 2:
            return Sentiment.NEGATIVE
        elif self.score == 3:
            return Sentiment.NEUTRAL
        else: #Score of 4 or 5
            return Sentiment.POSITIVE

class ReviewContainer:
    def __init__(self, reviews):
        self.reviews = reviews
        
    def get_text(self):
        return [x.text for x in self.reviews]
    
    def get_sentiment(self):
        return [x.sentiment for x in self.reviews]

    # Balances the dataset by undersampling the positive class to match the number of negative reviews
    def evenly_distribute(self):
        negative = list(filter(lambda x: x.sentiment == Sentiment.NEGATIVE, self.reviews))
        positive = list(filter(lambda x: x.sentiment == Sentiment.POSITIVE, self.reviews))
        # slices positive down to len(negative)
        positive_shrunk = positive[:len(negative)]
        self.reviews = negative + positive_shrunk
        # call this in place so positive and negative samples aren't grouped in blocks. Make sure that import random in imported
        random.shuffle(self.reviews)
        
        

### Load Data

In [3]:
import json

file_name = './data/sentiment/books_small_10000.json'

reviews = []
with open(file_name) as f:
    for line in f:
        review = json.loads(line)
        reviews.append(Review(review['reviewText'], review['overall']))
        
reviews[5].text
        

'I hoped for Mia to have some peace in this book, but her story is so real and raw.  Broken World was so touching and emotional because you go from Mia\'s trauma to her trying to cope.  I love the way the story displays how there is no "just bouncing back" from being sexually assaulted.  Mia showed us how those demons come for you every day and how sometimes they best you. I was so in the moment with Broken World and hurt with Mia because she was surrounded by people but so alone and I understood her feelings.  I found myself wishing I could give her some of my courage and strength or even just to be there for her.  Thank you Lizzy for putting a great character\'s voice on a strong subject and making it so that other peoples story may be heard through Mia\'s.'

### Prep Data

In [41]:
from sklearn.model_selection import train_test_split

# 33% is for testing, 66% is for training
training, test = train_test_split(reviews, test_size=0.33, random_state=42)

train_container = ReviewContainer(training)

test_container = ReviewContainer(test)


In [9]:
print(len(training))
print(len(test))

6700
3300


In [13]:
train_x = [x.text for x in training]
train_y = [x.sentiment for x in training]
print(train_x[0])
print(train_y[0])

test_x = [x.text for x in test]
test_y = [x.sentiment for x in test]
print(test_x[0])
print(test_y[0])

Olivia Hampton arrives at the Dunraven family home as cataloger of their extensive library. What she doesn't expect is a broken carriage wheel on the way. Nor a young girl whose mind is clearly gone, an old man in need of care himself (and doesn&#8217;t quite seem all there in Olivia&#8217;s opinion). Furthermore, Marion Dunraven, the only sane one of the bunch and the one Olivia is inexplicable drawn to, seems captive to everyone in the dusty old house. More importantly, she doesn't expect to fall in love with Dunraven's daughter Marion.Can Olivia truly believe the stories of sadness and death that surround the house, or are they all just local neighborhood rumor?Was that carriage trouble just a coincidence or a supernatural sign to stay away? If she remains, will the Castle&#8217;s dark shadows take Olivia down with them or will she and Marion long enough to declare their love?Patty G. Henderson has created an atmospheric and intriguing story in her Gothic tale. I found this to be an

In [ ]:
train_container.evenly_distribute()
train_x = train_container.get_text()
train_y = train_container.get_sentiment()

# Balancing test_container makes test evaluation treat both classes with equal sample counts
# (which is common in tutorials/assignments to inspect balanced accuracy/F1), 
# though in real-world deployment you usually leave test data in its natural distribution.
test_container.evenly_distribute()
test_x = test_container.get_text()
test_y = test_container.get_sentiment()

print(train_y.count(Sentiment.POSITIVE))
print(train_y.count(Sentiment.NEGATIVE))

436
436


#### Bag of words vectorization

In [44]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
train_x_vectors = vectorizer.fit_transform(train_x)
# The above line is equal to:
# vectorizer.fit(train_x)
# train_x_vectors = vectorizer.transform(train_x)

print(train_x[0])
print(train_x_vectors[0])
print(train_x_vectors[0].toarray())

I think I would have enjoyed the story if I could have gotten past the sexual nature of the crime.  I didn't read much beyond the discovery of the victim for that reason.  Not for me, but I will investigate another in the series because the detective sounds interesting.
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 35 stored elements and shape (1, 8906)>
  Coords	Values
  (0, 7968)	1
  (0, 8823)	1
  (0, 3669)	2
  (0, 2688)	1
  (0, 7929)	7
  (0, 7533)	1
  (0, 3968)	1
  (0, 1790)	1
  (0, 3455)	1
  (0, 5736)	1
  (0, 7031)	1
  (0, 5303)	1
  (0, 5478)	2
  (0, 1857)	1
  (0, 2230)	1
  (0, 6392)	1
  (0, 5224)	1
  (0, 878)	1
  (0, 2306)	1
  (0, 8502)	1
  (0, 3177)	2
  (0, 7925)	1
  (0, 6415)	1
  (0, 5408)	1
  (0, 4982)	1
  (0, 1168)	1
  (0, 8728)	1
  (0, 4235)	1
  (0, 449)	1
  (0, 4034)	1
  (0, 7002)	1
  (0, 797)	1
  (0, 2181)	1
  (0, 7334)	1
  (0, 4190)	1
[[0 0 0 ... 0 0 0]]


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# This book is great !
# This book was so bad
# -> 'This', 'is', 'book' is not so important, 'great' and 'bad' is much more important
# TF-IDF: TF(Term Frequency), IDF(Inverse Document Frequency) - how rare or common a word is across all documents in the training set
# Very common words like 'the', 'is', or 'book' appear in almost every document, so there IDF is low - reducing their weight
# In scikit-learn, TfidfVectorizer applies smoothing and L2 row normalization by default so every document vector has unit length
# Unit length: the L2 norm(Euclidean length) of a document's feature vector equals 1
# Scaling every row to length 1 shifts focus to the relative proportions/importance of words rather than sheer review length

vectorizer = TfidfVectorizer()
train_x_vectors = vectorizer.fit_transform(train_x)

# Never use .fit() on test_x because the vectorizer learns vocabulary words, word frequencies, or IDF weights from the test reviews.
# That leaks information from the test set into your feature representation -> no longer a fair measure
# A trained classifier expects input vectors with the exact same feature columns (vocabularies) it saw during training -> fitting a vectorizer separately on test_x creates a different vocabulary (some training words may be missing, new test words may appear in different positions, and the total number of columns can change)
# This book is great -> {"this": 0, "book": 1, "is": 2, "great": 3, "was": 4, "bad": 5} (this whole dictionary is the vocabulary)
 
test_x_vectors = vectorizer.transform(test_x)

print(train_x[0])
print(train_x_vectors[0].toarray())


I think I would have enjoyed the story if I could have gotten past the sexual nature of the crime.  I didn't read much beyond the discovery of the victim for that reason.  Not for me, but I will investigate another in the series because the detective sounds interesting.
[[0. 0. 0. ... 0. 0. 0.]]


## Classification

### Linear SVM



















In [46]:
from sklearn import svm

clf_svm = svm.SVC(kernel='linear')

clf_svm.fit(train_x_vectors, train_y)

clf_svm.predict(test_x_vectors[0])

array(['POSITIVE'], dtype='<U8')

### Decision Tree

In [47]:
from sklearn.tree import DecisionTreeClassifier

clf_dec = DecisionTreeClassifier()
clf_dec.fit(train_x_vectors, train_y)

clf_dec.predict(test_x_vectors[0])


array(['POSITIVE'], dtype='<U8')

### Naive Bayes

In [48]:
from sklearn.naive_bayes import GaussianNB

clf_gnb = GaussianNB()
#fit(X, y, sample_weight = None) - X: array
clf_gnb.fit(train_x_vectors.toarray(), train_y)

clf_gnb.predict(test_x_vectors[0].toarray())


array(['POSITIVE'], dtype='<U8')

### Logistic Regression

In [49]:
from sklearn.linear_model import LogisticRegression

clf_log = LogisticRegression()
clf_log.fit(train_x_vectors, train_y)

clf_log.predict(test_x_vectors[0])



array(['POSITIVE'], dtype='<U8')

### Random Forest

In [50]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = RandomForestClassifier()
clf_rf.fit(train_x_vectors, train_y)

clf_rf.predict(test_x_vectors[0])

array(['POSITIVE'], dtype='<U8')

## Evaluation

In [51]:
# Mean Accuracy on the give test data and labels
print(clf_svm.score(test_x_vectors, test_y))
print(clf_dec.score(test_x_vectors, test_y))
print(clf_gnb.score(test_x_vectors.toarray(), test_y))
print(clf_log.score(test_x_vectors, test_y))
print(clf_rf.score(test_x_vectors, test_y))

0.8076923076923077
0.6298076923076923
0.6610576923076923
0.8028846153846154
0.7932692307692307


In [54]:
# F1 Scores: can be interpreted as a weighted average of the precision and recall, where an F1 score reaches its best value at 1 and worst score at 0.
# precision: out of all samples predicted as positive, how many are actually positive?
# recall: out of all actual positive samples, how many did the model catch
from sklearn.metrics import f1_score

f1_score(test_y, clf_svm.predict(test_x_vectors), average=None, labels=[Sentiment.POSITIVE, Sentiment.NEGATIVE])
#f1_score(test_y, clf_log.predict(test_x_vectors), average=None, labels=[Sentiment.POSITIVE, Sentiment.NEUTRAL, Sentiment.NEGATIVE])

# ex: [0.92, 0.093, 0.1] -> predict positive very good but neutral and negative are trash

array([0.80582524, 0.80952381])

In [34]:
print(train_y[0:5]) # there are many positives
print(train_y.count(Sentiment.POSITIVE)) # to see how many positives there are -> heavily biased
print(train_y.count(Sentiment.NEGATIVE))

['POSITIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE']
5611
436


In [56]:
test_set = ['very fun', "bad book do not buy", 'horrible waste of time']
new_test = vectorizer.transform(test_set)

clf_svm.predict(new_test)


array(['POSITIVE', 'NEGATIVE', 'NEGATIVE'], dtype='<U8')

### Tuning our model (with Grid Search)

In [58]:
from sklearn.model_selection import GridSearchCV

parameters = {'kernel': ('linear', 'rbf'), 'C': (1,4,8,16,32)}

svc = svm.SVC()
clf = GridSearchCV(svc, parameters, cv=5)
clf.fit(train_x_vectors, train_y)
















C:\ProgramData\Anaconda3\lib\site-packages\sklearn\svm\base.py:193: FutureWarning: The default value of gamma will change from 'auto' to 'scale' in version 0.22 to account better for unscaled features. Set gamma explicitly to 'auto' or 'scale' to avoid this warning.
  "avoid this warning.", FutureWarning)


GridSearchCV(cv=5, error_score='raise-deprecating',
             estimator=SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
                           decision_function_shape='ovr', degree=3,
                           gamma='auto_deprecated', kernel='rbf', max_iter=-1,
                           probability=False, random_state=None, shrinking=True,
                           tol=0.001, verbose=False),
             iid='warn', n_jobs=None,
             param_grid={'C': (1, 4, 8, 16, 32), 'kernel': ('linear', 'rbf')},
             pre_dispatch='2*n_jobs', refit=True, return_train_score=False,
             scoring=None, verbose=0)

In [66]:
print(clf.score(test_x_vectors, test_y))

0.8076923076923077


## Saving Model

#### Save model

In [68]:
import pickle

with open('./models/sentiment_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

#### Load model

In [70]:
with open('./models/entiment_classifier.pkl', 'rb') as f:
    loaded_clf = pickle.load(f)

In [72]:
print(test_x[0])

loaded_clf.predict(test_x_vectors[0])

I loved this book and the previous books in this series. It brings out every emotion you can think of. I look forward to reading more books by this author.


array(['POSITIVE'], dtype='<U8')

In [ ]:
from sklearn.linear_model import Perceptron

clf = Perceptron(tol=1e-3, random_state=0)
clf.fit(X, y)  